# Build the Silver demand table

This notebook applies one parser to the monthly, daily and current Bronze CSVs. PySpark handles the irregular AEMO row structure; Spark SQL then combines the three clean views and writes one Delta table.

### Storage prerequisite

Before running the pipeline, I created the Databricks Volume `/Volumes/workspace/default/aemo_mlops_volume`. The ingestion notebooks write their Bronze files there, and the Silver transformation reads from the same location.

## 1. Import the Spark transformations

Only Spark DataFrame functions and a window definition are required by the current implementation.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 2. Parse `DISPATCH,REGIONSUM`

AEMO files are not ordinary header-first CSVs. The `I` row defines the column positions and the `D` rows contain measurements. Looking up positions from the header keeps the parser aligned with the published schema.

In [0]:
def parse_dispatchregionsum(path):
    # Read each physical line as text so the AEMO record type can be inspected first.
    raw = spark.read.text(path)

    # The I row describes the columns for the DISPATCH,REGIONSUM section.
    header = (
        raw
        .filter(F.col("value").startswith("I,DISPATCH,REGIONSUM"))
        .select(F.split("value", ",").alias("fields"))
        .first()["fields"]
    )

    # Resolve positions by name instead of assuming fixed numeric indexes.
    time_i = header.index("SETTLEMENTDATE")
    runno_i = header.index("RUNNO")
    region_i = header.index("REGIONID")
    intervention_i = header.index("INTERVENTION")
    demand_i = header.index("TOTALDEMAND")

    df = (
        raw
        # Keep only measurement rows from the required section.
        .filter(F.col("value").startswith("D,DISPATCH,REGIONSUM"))
        .withColumn("fields", F.split("value", ","))
        # Select the standard operational observation for Victoria.
        .filter(F.col("fields")[region_i] == "VIC1")
        .filter(F.col("fields")[runno_i].cast("int") == 1)
        .filter(F.col("fields")[intervention_i].cast("int") == 0)
        .select(
            # Convert AEMO's timestamp text into a Spark timestamp.
            F.to_timestamp(
                F.regexp_replace(
                    F.col("fields")[time_i],
                    '"',
                    ''
                ),
                "yyyy/MM/dd HH:mm:ss"
            ).alias("time"),

            # TOTALDEMAND becomes the numeric demand value used downstream.
            F.col("fields")[demand_i]
            .cast("double")
            .alias("demand")
        )
    )

    return df

## 3. Apply the same parser to every Bronze source

The archive shapes differ during ingestion, but their extracted `DISPATCH,REGIONSUM` rows share the same schema. Each source therefore becomes a two-column `time` and `demand` DataFrame.

In [0]:
base_path = "/Volumes/workspace/default/aemo_mlops_volume/bronze"

monthly = parse_dispatchregionsum(
    f"{base_path}/monthly_uncompressed/*.CSV"
)

daily = parse_dispatchregionsum(
    f"{base_path}/daily_uncompressed/*.CSV"
)

current = parse_dispatchregionsum(
    f"{base_path}/current_uncompressed/*.CSV"
)

## 4. Expose the DataFrames to Spark SQL

Temporary views provide a clear boundary: PySpark prepares each source, then SQL expresses how the three sources are combined.

In [0]:
monthly.createOrReplaceTempView("monthly")
daily.createOrReplaceTempView("daily")
current.createOrReplaceTempView("current")

## 5. Combine, deduplicate and write Delta

`UNION ALL` retains every candidate row. `ROW_NUMBER` then keeps one row for each timestamp according to the numeric source priority before the result is written as `demand_vic_5min`.

In [0]:
%sql

-- Rebuild the Silver table from the three point-in-time Bronze views.
CREATE OR REPLACE TABLE workspace.default.demand_vic_5min
USING DELTA
AS

WITH all_data AS (

    -- Higher numeric priority wins when the same timestamp exists in multiple sources.
    SELECT time, demand, 3 AS priority
    FROM monthly

    UNION ALL

    SELECT time, demand, 2 AS priority
    FROM daily

    UNION ALL

    SELECT time, demand, 1 AS priority
    FROM current
),

deduplicated AS (

    SELECT
        time,
        demand,
        ROW_NUMBER() OVER (
            PARTITION BY time
            ORDER BY priority DESC
        ) AS rn

    FROM all_data
)

SELECT
    time,
    demand

FROM deduplicated

WHERE rn = 1;

## 6. Inspect the result

Ordering the table by time provides a final visual check that Silver contains the expected chronological demand series.

In [0]:
%sql

SELECT *
FROM workspace.default.demand_vic_5min
ORDER BY time;